In [1]:
from dotenv import load_dotenv
import requests
import json
import os

load_dotenv()

True

In [2]:
url = "https://realtime.oxylabs.io/v1/queries"
auth = (os.environ.get("USERNAME"), os.environ.get("PASSWORD"))

def get_guery(keyword, date_from="", date_to=""):

    payload = {
        "source": "google_trends_explore",
        "query": keyword,
        "context":[
            {"key":"date_from", "value":date_from},
            {"key":"date_to", "value":date_to},
        ]
    }

    try:
        response = requests.request("POST", url, auth=auth, json=payload, timeout=180)
    except requests.exceptions.RequestException as e:
        print("Caught exception while getting trend data")
        raise e

    data = response.json()
    content = data["results"][0]["content"]

    return json.loads(content)

In [3]:
from datetime import datetime, timedelta, date

date_from = datetime(month=5, day=1, year=2022)
date_to = date_from + timedelta(days=180) #
final_date_time = datetime(month=12, day=5, year=2025)

final_dict = {
    "cemantix" : [],
    "cémantix" : [],
    "pedantix" : [],
    "pédantix" : [],
}

while date_to < final_date_time + timedelta(days=180):

    print(f"Current dates : {date_from.date()} - {date_to.date()}")

    for key, value in final_dict.items():

        trend_data = get_guery(key, date_from=str(date_from.date()), date_to=str(date_to.date()))
        value += trend_data["interest_over_time"][0]["items"]

    date_from = date_from + timedelta(days=180)
    date_to = date_to + timedelta(days=180)

Current dates : 2022-05-01 - 2022-10-28
Current dates : 2022-10-28 - 2023-04-26
Current dates : 2023-04-26 - 2023-10-23
Current dates : 2023-10-23 - 2024-04-20
Current dates : 2024-04-20 - 2024-10-17
Current dates : 2024-10-17 - 2025-04-15
Current dates : 2025-04-15 - 2025-10-12
Current dates : 2025-10-12 - 2026-04-10


In [77]:
import pandas as pd

dict_for_pandas = {key:[data["value"] for data in value] for key, value in final_dict.items()}
dict_for_pandas["date"] = pd.to_datetime([datetime.strptime(data["time"], "%b %d, %Y").date() for data in final_dict["cemantix"]])

In [89]:
df_google = pd.DataFrame(dict_for_pandas).drop_duplicates(subset='date', keep="first")
df_google

,cemantix,cémantix,pedantix,pédantix,date
0,71,46,0,0,2022-05-01
1,49,55,0,0,2022-05-02
2,51,59,0,0,2022-05-03
3,81,95,0,0,2022-05-04
4,56,51,0,0,2022-05-05
...,...,...,...,...,...
1317,100,100,84,83,2025-12-01
1318,84,65,87,83,2025-12-02
1319,99,76,75,74,2025-12-03
1320,94,68,78,74,2025-12-04


In [90]:
df_winners = pd.read_csv('solvers.txt', sep="\t", header = None)
df_winners.columns = ['number', 'winners', 'topic']
today = datetime(year = 2025, month = 12, day = 3).date()
today_nr = 1300
date_col = [today - timedelta(1300 - number) for number in df_winners.number]
df_winners['date'] = pd.to_datetime(date_col)
df_winners

,number,winners,topic,date
0,1,4398,Affaire Dreyfus,2022-05-14
1,2,6632,Nestlé,2022-05-15
2,3,9224,Formation et évolution du Système solaire,2022-05-16
3,4,13865,Strasbourg,2022-05-17
4,5,11130,Processeur,2022-05-18
...,...,...,...,...
1270,1271,33282,Premier Empire bulgare,2025-11-04
1271,1272,36303,Violon,2025-11-05
1272,1273,38471,Fée,2025-11-06
1273,1274,23664,Urartu,2025-11-07


In [94]:
merged_dataframe = pd.merge(df_winners, df_google, how='inner', on='date')
merged_dataframe.set_index("date", inplace=True)
merged_dataframe

,number,winners,topic,cemantix,cémantix,pedantix,pédantix
date,,,,,,,
2022-05-14,1,4398,Affaire Dreyfus,76,46,0,0
2022-05-15,2,6632,Nestlé,57,48,19,0
2022-05-16,3,9224,Formation et évolution du Système solaire,67,73,12,0
2022-05-17,4,13865,Strasbourg,67,77,27,0
2022-05-18,5,11130,Processeur,67,73,42,50
...,...,...,...,...,...,...,...
2025-11-04,1271,33282,Premier Empire bulgare,79,64,62,65
2025-11-05,1272,36303,Violon,79,69,57,44
2025-11-06,1273,38471,Fée,72,60,59,62


In [95]:
merged_dataframe.to_csv("cleaned_data.csv", index=True)